<a href="https://colab.research.google.com/github/emmanuelraju671-prog/deconstructing-emotion-ai/blob/main/System_2_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets tqdm

In [ ]:
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score
tqdm.pandas()

print("1. Downloading GoEmotions Test data...")
dataset = load_dataset("google-research-datasets/go_emotions", split="test")
df = pd.DataFrame(dataset)

df['primary_emotion_id'] = df['labels'].apply(lambda x: x[0] if len(x) > 0 else 27)
emotion_mapping = dataset.features['labels'].feature.names
df['true_emotion_text'] = df['primary_emotion_id'].apply(lambda x: emotion_mapping[x])

df_sample = df.sample(n=500, random_state=42).copy()

print("2. Loading RoBERTa Transformer Model...")
classifier = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions")

print("3. Analyzing emotional context...")
def get_emotion(text):
    result = classifier(text)[0]
    return result['label']

df_sample['predicted_emotion'] = df_sample['text'].progress_apply(get_emotion)

print("\n4. Calculating System 2 Accuracy...")
accuracy = accuracy_score(df_sample['true_emotion_text'], df_sample['predicted_emotion'])

print(f"\n======================================")
print(f"System 2 (Transformer) Accuracy: {accuracy * 100:.2f}%")
print(f"======================================")